<a href="https://colab.research.google.com/github/Rajeraghav/AI-Engineer-Journey/blob/main/RNN_IMDB_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================================
# IMDB Movie Review Sentiment Analysis using Simple RNN
# Runtime CSV Upload Version
# ==========================================================

import pandas as pd
import numpy as np

from google.colab import files
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# -----------------------------
# Upload CSV at runtime
# -----------------------------
uploaded = files.upload()

filename = list(uploaded.keys())[0]

# Read dataset
df = pd.read_csv(filename)

# -----------------------------
# Display dataset
# -----------------------------
print("Dataset Shape :", df.shape)
print(df.head())

# -----------------------------
# Keep required columns
# -----------------------------
df = df[['review', 'sentiment']]

# Remove missing values
df.dropna(inplace=True)

# -----------------------------
# Convert labels
# positive -> 1
# negative -> 0
# -----------------------------
df['sentiment'] = df['sentiment'].map({
    'positive':1,
    'negative':0
})

# -----------------------------
# Text Tokenization
# -----------------------------
max_words = 10000
max_length = 200

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(df['review'])

sequences = tokenizer.texts_to_sequences(df['review'])

X = pad_sequences(
    sequences,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

y = df['sentiment'].values

# -----------------------------
# Train Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# -----------------------------
# Build Simple RNN Model
# -----------------------------
model = Sequential([
    Embedding(input_dim=max_words, output_dim=64),
    SimpleRNN(64),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# -----------------------------
# Train Model
# -----------------------------
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

# -----------------------------
# Evaluate
# -----------------------------
loss, accuracy = model.evaluate(X_test, y_test)

print("\nTest Accuracy :", round(accuracy*100,2), "%")

# -----------------------------
# Prediction Function
# -----------------------------
def predict_sentiment(text):

    seq = tokenizer.texts_to_sequences([text])

    pad = pad_sequences(seq, maxlen=max_length,
                        padding='post',
                        truncating='post')

    pred = model.predict(pad, verbose=0)[0][0]

    if pred >= 0.5:
        print("Positive Review")
    else:
        print("Negative Review")

    print("Probability :", round(float(pred),4))

# -----------------------------
# Example Prediction
# -----------------------------
sample = "This movie was absolutely fantastic and I loved it."

predict_sentiment(sample)

Saving IMDB-Dataset.csv to IMDB-Dataset.csv
Dataset Shape : (50000, 2)
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 40s 74ms/step - accuracy: 0.5034 - loss: 0.6937 - val_accuracy: 0.5000 - val_loss: 0.6940
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 38s 76ms/step - accuracy: 0.5090 - loss: 0.6951 - val_accuracy: 0.5026 - val_loss: 0.7002
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 33s 67ms/step - accuracy: 0.5200 - loss: 0.6920 - val_accuracy: 0.5084 - val_loss: 0.6941
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 43s 70ms/step - accuracy: 0.5564 - loss: 0.6811 - val_accuracy: 0.5203 - val_loss: 0.6937
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━